In [ ]:
import os
import sys

sys.path.append(os.path.abspath(".."))

import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV
from src.models import get_models
from src.tuning import get_tuning_models, get_tuning_params
from src.preprocessing import RANDOM_STATE, get_preprocessor, load_data, split_data

print("Import OK")

Import OK


In [2]:
df = load_data()

X_train, X_test, y_train, y_test = split_data(df)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (4088, 10)
X_test: (1022, 10)
y_train: (4088,)
y_test: (1022,)


In [3]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = [
    "recall",
    "precision",
    "f1",
    "roc_auc",
    "average_precision"
]

print("Đã tạo 5-Fold Cross Validation")
print("Scoring:", scoring)

Đã tạo 5-Fold Cross Validation
Scoring: ['recall', 'precision', 'f1', 'roc_auc', 'average_precision']


In [4]:
rows = []

for name, model in get_models().items():
    pipe = Pipeline([
        ("preprocessor", get_preprocessor()),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("model", model),
    ])

    res = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    row = {"model": name}

    for m in scoring:
        row[f"{m}_mean"] = res[f"test_{m}"].mean()
        row[f"{m}_std"] = res[f"test_{m}"].std()

    rows.append(row)

    print("Xong:", name)

cv_results = pd.DataFrame(rows).round(3)

cv_results

Xong: Logistic Regression
Xong: Decision Tree
Xong: KNN
Xong: SVM
Xong: Random Forest
Xong: XGBoost


,model,recall_mean,recall_std,precision_mean,precision_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std
0,Logistic Regression,0.754,0.039,0.131,0.009,0.223,0.015,0.836,0.011,0.192,0.036
1,Decision Tree,0.221,0.040,0.120,0.020,0.155,0.026,0.569,0.019,0.065,0.007
2,KNN,0.371,0.067,0.113,0.022,0.173,0.033,0.671,0.026,0.088,0.011
3,SVM,0.458,0.040,0.116,0.008,0.185,0.012,0.779,0.018,0.137,0.030
4,Random Forest,0.085,0.025,0.122,0.034,0.100,0.028,0.788,0.012,0.123,0.005
5,XGBoost,0.106,0.037,0.168,0.085,0.129,0.053,0.799,0.009,0.143,0.012


In [6]:
cv_results.to_csv("../results/cv_results.csv", index=False)

print("Đã lưu kết quả vào: ../results/cv_results.csv")

Đã lưu kết quả vào: ../results/cv_results.csv


In [5]:
# Đọc lại kết quả CV đã lưu
cv_results = pd.read_csv("../results/cv_results.csv")

# Chọn 2 mô hình tốt nhất theo Average Precision
top_2_models = (
    cv_results
    .sort_values("average_precision_mean", ascending=False)
    .head(2)
)

top_2_models[[
    "model",
    "average_precision_mean",
    "roc_auc_mean",
    "f1_mean",
    "recall_mean",
    "precision_mean"
]]

,model,average_precision_mean,roc_auc_mean,f1_mean,recall_mean,precision_mean
0,Logistic Regression,0.192,0.836,0.223,0.754,0.131
5,XGBoost,0.143,0.799,0.129,0.106,0.168


In [6]:
print("2 mô hình được chọn theo Average Precision:")
for _, row in top_2_models.iterrows():
    print(f"- {row['model']}: AP = {row['average_precision_mean']:.3f}")

2 mô hình được chọn theo Average Precision:
- Logistic Regression: AP = 0.192
- XGBoost: AP = 0.143


In [7]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

from src.tuning import get_tuning_models

In [8]:
rf_pipe = Pipeline([
    ("preprocessor", get_preprocessor()),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("model", get_tuning_models()["Random Forest"]),
])

print("Random Forest pipeline OK")

Random Forest pipeline OK


In [2]:
tuning_params = get_tuning_params()

rf_params = tuning_params["Random Forest"]
xgb_params = tuning_params["XGBoost"]

print("Tuning parameters loaded from src.tuning")

Tuning parameters loaded from src.tuning


In [10]:
rf_search = RandomizedSearchCV(
    estimator=rf_pipe,
    param_distributions=rf_params,
    n_iter=20,
    scoring="average_precision",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X_train, y_train)

print("Random Forest tuning xong!")
print("Best score:", rf_search.best_score_)
print("Best params:", rf_search.best_params_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Random Forest tuning xong!
Best score: 0.15630700467257658
Best params: {'model__max_depth': 5, 'model__max_features': 'log2', 'model__min_samples_leaf': 2, 'model__min_samples_split': 6, 'model__n_estimators': 290}


In [11]:
xgb_pipe = Pipeline([
    ("preprocessor", get_preprocessor()),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("model", get_tuning_models()["XGBoost"]),
])

print("XGBoost pipeline OK")

XGBoost pipeline OK


In [3]:
xgb_search = RandomizedSearchCV(
    estimator=xgb_pipe,
    param_distributions=xgb_params,
    n_iter=20,
    scoring="average_precision",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(X_train, y_train)

print("XGBoost tuning xong!")
print("Best score:", xgb_search.best_score_)
print("Best params:", xgb_search.best_params_)

NameError: name 'RandomizedSearchCV' is not defined

In [14]:
tuning_results = pd.DataFrame([
    {
        "model": "Random Forest",
        "best_cv_average_precision": rf_search.best_score_
    },
    {
        "model": "XGBoost",
        "best_cv_average_precision": xgb_search.best_score_
    }
]).round(4)

tuning_results

,model,best_cv_average_precision
0,Random Forest,0.1563
1,XGBoost,0.1606


In [15]:
final_model = xgb_search.best_estimator_

print("Final model:", final_model)

Final model: Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'avg_glucose_level',
                                                   'bmi']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['gender', 'ever_married',
                                                   'work_type',
                                        

In [16]:
from sklearn.model_selection import cross_val_predict

oof_proba = cross_val_predict(
    final_model,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

print("OOF probabilities:", oof_proba.shape)

OOF probabilities: (4088,)


In [17]:
from sklearn.metrics import f1_score
import numpy as np

thresholds = np.arange(0.10, 0.91, 0.01)

threshold_results = []

for threshold in thresholds:
    y_pred = (oof_proba >= threshold).astype(int)
    threshold_results.append({
        "threshold": threshold,
        "f1": f1_score(y_train, y_pred),
    })

threshold_results = pd.DataFrame(threshold_results)

best_threshold = threshold_results.loc[
    threshold_results["f1"].idxmax(), "threshold"
]

best_f1 = threshold_results["f1"].max()

print("Best threshold:", best_threshold)
print("Best CV F1:", round(best_f1, 4))

Best threshold: 0.5699999999999997
Best CV F1: 0.2295


In [18]:
final_model.fit(X_train, y_train)

print("Final model fitted on full training set.")

Final model fitted on full training set.


In [19]:
y_test_proba = final_model.predict_proba(X_test)[:, 1]

y_test_pred = (y_test_proba >= 0.57).astype(int)

print("Test prediction done.")
print("Threshold:", 0.57)

Test prediction done.
Threshold: 0.57


In [20]:
from sklearn.metrics import (
    recall_score,
    precision_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

test_results = {
    "recall": recall_score(y_test, y_test_pred),
    "precision": precision_score(y_test, y_test_pred),
    "f1": f1_score(y_test, y_test_pred),
    "roc_auc": roc_auc_score(y_test, y_test_proba),
    "average_precision": average_precision_score(y_test, y_test_proba)
}

test_results = pd.DataFrame([test_results]).round(4)

test_results

,recall,precision,f1,roc_auc,average_precision
0,0.68,0.156,0.2537,0.8355,0.2606


In [21]:
test_results.to_csv("../results/test_results.csv", index=False)

print("Saved:", "../results/test_results.csv")

Saved: ../results/test_results.csv


In [22]:
tuning_results.to_csv("../results/tuning_results.csv", index=False)

print("Saved:", "../results/tuning_results.csv")

Saved: ../results/tuning_results.csv
